In [115]:
import jax

jax.config.update("jax_enable_x64", True)

from jax import numpy as jnp

In [116]:
import numpy as np

In [117]:
from ase.io import read, write

In [118]:
image = read("test.xyz")

image

Atoms(symbols='H4C', pbc=False)

In [119]:
xyz_np = image.positions

xyz_jx = jnp.array(xyz_np, dtype=jnp.float64)

xyz_np, xyz_jx

(array([[ 0.524051 , -0.9289334, -0.1644923],
        [-0.7560851,  0.0942945,  0.6223003],
        [ 0.0635526,  0.7348345, -0.6757111],
        [-1.1460125, -0.5418034, -1.1115551],
        [-0.3708009, -0.1384256, -0.2623446]]),
 Array([[ 0.524051 , -0.9289334, -0.1644923],
        [-0.7560851,  0.0942945,  0.6223003],
        [ 0.0635526,  0.7348345, -0.6757111],
        [-1.1460125, -0.5418034, -1.1115551],
        [-0.3708009, -0.1384256, -0.2623446]], dtype=float64))

In [120]:
from scipy.spatial import distance

from msa import basis, gradient
from pes_utils import pes_utils

In [121]:
_ALPHA = 2.0  # Angstrom

In [122]:
def morse(
    r: np.ndarray,
    alpha: float = 1.0,
) -> np.ndarray:
    return np.exp(-r / alpha)

In [123]:
r = distance.pdist(xyz_np)
y = morse(r, alpha=_ALPHA)

r, y

(array([1.81790714, 1.80042413, 1.95854785, 1.19801401, 1.66340946,
        1.88756946, 0.99257199, 1.81185321, 1.05930073, 1.21853402]),
 array([0.40294566, 0.40648345, 0.3755837 , 0.54935687, 0.43530657,
        0.38915221, 0.60878751, 0.40416721, 0.5888108 , 0.54374929]))

In [124]:
mono_msa = basis.evmono(y)
poly_msa = basis.evpoly(mono_msa)

In [125]:
p_msa = basis.bemsav(y)

p_msa

array([1.        , 2.29070447, 2.4136388 , 1.96627516, 2.76148645,
       0.48453541, 2.76744673, 1.94182097, 1.31477665, 0.9729395 ,
       0.74957303, 0.78927837, 3.16390139, 1.10992742, 0.79269824,
       1.10964093, 2.22408164, 0.77961662, 0.26036317, 1.11441539,
       0.26036443, 2.2554362 , 1.58329121, 1.59010473, 1.1119399 ,
       0.38987684, 1.11677696, 1.56543836, 0.75632855, 0.39300933,
       0.10707665, 0.90362022, 0.63514783, 0.90557833, 0.3175821 ,
       1.2699855 , 0.63635913, 0.89197956, 0.07824978, 0.14862317,
       0.63634507, 1.27546462, 0.89389172, 0.4477919 , 0.44683799,
       0.31359008, 0.14957998, 1.2887437 , 0.90438328, 1.81264015,
       1.81652581, 0.63705584, 0.91025907, 0.63552887, 1.27654537,
       0.64098663, 0.3174653 , 0.4460584 , 1.27536097, 0.44703421,
       0.3202405 , 0.89358533, 0.89551827, 0.62729094, 0.89748041,
       0.31424272, 0.31483257, 0.89937265, 0.31483563, 0.64629071,
       1.29646896, 0.90982881, 0.91567512, 0.63683414, 0.07827

In [126]:
assert np.allclose(p_msa, poly_msa)

In [127]:
from jaxpip.descriptor import PolynomialDescriptor

In [128]:
pip_a4b_4 = PolynomialDescriptor.from_file(
    basis_file="MOL_4_1_4.json",
    alpha=_ALPHA,
    with_grad=True,
    dtype=jnp.float64,
)

pip_a4b_4

PolynomialDescriptor(
  num_flat_exponents=1001,
  num_atoms=5,
  num_pips=83,
  dtype=<class 'jax.numpy.float64'>,
  with_grad=True
)

In [129]:
p_jx, J_p_xyz_jx = pip_a4b_4(xyz_jx)

p_jx.shape, J_p_xyz_jx.shape

((83,), (83, 5, 3))

In [130]:
np.allclose(p_msa, p_jx)

True

In [131]:
drdx_msa = pes_utils.evdrdx(xyz_np.T)

drdx_msa.shape

(15, 10)

In [132]:
J_p_xyz_msa = np.array([gradient.dbemsav(drdx_msa, mono_msa, poly_msa, j)
                       for j in range(1, 3 * len(xyz_np) + 1)])

J_p_xyz_msa.shape  # (3 * 5, 83)

(15, 83)

In [133]:
%%timeit

p_msa = basis.bemsav(y)

J_p_xyz_msa = np.array([gradient.dbemsav(drdx_msa, mono_msa, poly_msa, j)
                       for j in range(1, 3 * len(xyz_np) + 1)])

10.4 μs ± 281 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [134]:
%%timeit

p_jx, J_p_xyz_jx = pip_a4b_4(xyz_jx)

p_jx.block_until_ready()
J_p_xyz_jx.block_until_ready()

62.8 μs ± 5.75 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [135]:
J_p_xyz_jx[2, :, :].flatten()

Array([-0.35398801,  0.33833556, -0.06131889,  0.20892624, -0.0951587 ,
       -0.43577031, -0.190172  , -0.41401719,  0.17893945,  0.33523377,
        0.17084033,  0.31814975,  0.        ,  0.        ,  0.        ],      dtype=float64)

In [136]:
J_p_xyz_msa[:, 2]

array([-0.35398801,  0.33833556, -0.06131889,  0.20892624, -0.0951587 ,
       -0.43577031, -0.190172  , -0.41401719,  0.17893945,  0.33523377,
        0.17084033,  0.31814975, -0.        , -0.        , -0.        ])

In [137]:
for i in range(3 * len(xyz_np)):
    residue = J_p_xyz_jx[i, :, :].flatten() - J_p_xyz_msa[:, i]

    print(f"J_p_xyz_jx[{i}, :, :].flatten() - J_p_xyz_msa[:, {i}] = ")
    print(residue)

J_p_xyz_jx[0, :, :].flatten() - J_p_xyz_msa[:, 0] = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
J_p_xyz_jx[1, :, :].flatten() - J_p_xyz_msa[:, 1] = 
[ 2.77555756e-17  0.00000000e+00  0.00000000e+00  2.77555756e-17
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  1.38777878e-17  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -2.77555756e-17  0.00000000e+00 -5.55111512e-17]
J_p_xyz_jx[2, :, :].flatten() - J_p_xyz_msa[:, 2] = 
[-5.55111512e-17  5.55111512e-17 -2.77555756e-17  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  5.55111512e-17 -2.77555756e-17  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00]
J_p_xyz_jx[3, :, :].flatten() - J_p_xyz_msa[:, 3] = 
[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  2.77555756e-17
  0.00000000e+00  0.00000000e+00 -2.77555756e-17 -5.55111512e-17
  2.77555756e-17  0.00000000e+00  2.77555756e-17  0.00000000e+00
 -3.46944695e-17 -2.77555756e-17 -5.55111512e-17]
J_p_xy

In [158]:
J_jx_aligned = J_p_xyz_jx.reshape(83, 15).T

is_matched = np.allclose(J_jx_aligned, J_p_xyz_msa, atol=1e-15)

is_matched, jnp.abs(J_jx_aligned - J_p_xyz_msa).max()

(True, Array(3.55271368e-15, dtype=float64))

In [159]:
from molpipx.pip_flax import PIP
from molpipx import get_functions, detect_molecule

from molpipx.utils_gradients import get_pip_grad

In [ ]:
molecule_type = "A4B"
na = 5
poly_degree = 4

In [160]:
f_mono, f_poly = get_functions(molecule_type, poly_degree)

In [161]:
pip_a4b_4_mpx = PIP(
    f_mono=f_mono,
    f_poly=f_poly,
    l=0.5,
)

params = pip_a4b_4_mpx.init(jax.random.PRNGKey(0), xyz_jx)

In [162]:
pip_a4b_4_mpx.l

0.5

In [163]:
p_mpx = pip_a4b_4_mpx.apply(params, xyz_jx)

p_mpx

Array([1.        , 1.31477665, 0.9729395 , 0.64629071, 0.63683414,
       0.078275  , 0.642364  , 0.3154874 , 0.43605623, 0.15908646,
       0.14077543, 0.10389931, 0.41917904, 0.10291414, 0.10572341,
       0.10280686, 0.20740738, 0.05075846, 0.01704705, 0.10458123,
       0.01704771, 0.42740165, 0.210317  , 0.21393933, 0.10366585,
       0.02539838, 0.10549732, 0.10314895, 0.1459149 , 0.02623417,
       0.01146541, 0.06818465, 0.0337233 , 0.06878132, 0.01686511,
       0.06738371, 0.03398215, 0.03322571, 0.00204146, 0.00553133,
       0.03397621, 0.0685545 , 0.03351032, 0.01688174, 0.01674008,
       0.00822742, 0.00567385, 0.13922661, 0.06841973, 0.13800963,
       0.13918524, 0.0341323 , 0.07022135, 0.03380813, 0.06879186,
       0.03497025, 0.01683892, 0.01662248, 0.06853431, 0.01677072,
       0.01744287, 0.03346507, 0.03375616, 0.01646736, 0.03405901,
       0.00829689, 0.00835834, 0.0343376 , 0.00835898, 0.07044601,
       0.14259248, 0.07009052, 0.07187585, 0.03408529, 0.00204

In [164]:
jnp.allclose(p_mpx, p_msa)

Array(False, dtype=bool)

In [ ]:
get_pip_grad()

In [149]:
%%timeit

p_mpx, J_p_xyz = molpipx_eval_p_and_J_p_xyz(xyz_jx)

p_mpx.block_until_ready()
J_p_xyz.block_until_ready()

2.51 ms ± 363 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [150]:
jaxpip_lowered = jax.jit(pip_a4b_4).lower(xyz_jx)
jaxpip_hlo = jaxpip_lowered.as_text()
num_ops_jaxpip = jaxpip_hlo.count('\n')

num_ops_jaxpip

139

In [151]:
molpipx_lowered = jax.jit(molpipx_eval_p_and_J_p_xyz).lower(xyz_jx)
molpipx_hlo = molpipx_lowered.as_text()
num_ops_molpipx = molpipx_hlo.count('\n')

num_ops_molpipx

7196

In [154]:
pip_a4b_4_no_grad = PolynomialDescriptor.from_file(
    basis_file="MOL_4_1_4.json",
    alpha=_ALPHA,
    with_grad=False,
    dtype=jnp.float64,
)

pip_a4b_4

PolynomialDescriptor(
  num_flat_exponents=1001,
  num_atoms=5,
  num_pips=83,
  dtype=<class 'jax.numpy.float64'>,
  with_grad=True
)

In [155]:
jaxpip_no_grad_lowered = jax.jit(pip_a4b_4_no_grad).lower(xyz_jx)
jaxpip_no_grad_hlo = jaxpip_no_grad_lowered.as_text()
num_ops_jaxpip_no_grad = jaxpip_no_grad_hlo.count('\n')

num_ops_jaxpip_no_grad

56

In [156]:
molpipx_lowered_no_grad = jax.jit(molpipx_eval).lower(xyz_jx)
molpipx_hlo_no_grad = molpipx_lowered_no_grad.as_text()
num_ops_molpipx_no_grad = molpipx_hlo_no_grad.count('\n')

num_ops_molpipx_no_grad

1862